# Тестирование API инструментов GraphArchitect

Этот notebook тестирует все API инструменты:
- OpenRouter (универсальный роутер LLM)
- OpenAI (GPT-4, GPT-3.5)
- DeepSeek (специализированный для кода)
- VLLM (локальные модели)
- Replicate (LLM, Audio, VQA)
- Infinity (эмбеддинги)

In [1]:
# Настройка окружения
import sys
from pathlib import Path

# Добавляем GraphArchitect в путь
grapharchitect_path = Path.cwd().parent.parent / "src" / "GraphArchitectLib"
sys.path.insert(0, str(grapharchitect_path))

import os

# Проверка API ключей
print("Проверка API ключей:")
print(f"  OPENROUTER_API_KEY: {'✓' if os.getenv('OPENROUTER_API_KEY') else '✗'}")
print(f"  OPENAI_API_KEY: {'✓' if os.getenv('OPENAI_API_KEY') else '✗'}")
print(f"  REPLICATE_API_TOKEN: {'✓' if os.getenv('REPLICATE_API_TOKEN') else '✗'}")

Проверка API ключей:
  OPENROUTER_API_KEY: ✗
  OPENAI_API_KEY: ✗
  REPLICATE_API_TOKEN: ✗


## 1. OpenRouter

Универсальный роутер для доступа к GPT-4, Claude, Gemini и другим моделям.

In [2]:
from grapharchitect.tools.ApiTools.OpenRouterTool.openrouter_llm import OpenRouterLLM

# Создание клиента
openrouter = OpenRouterLLM(
    model_name="openai/gpt-3.5-turbo",
    system_prompt="You are a helpful AI assistant."
)

print("OpenRouter инициализирован")
print(f"Модель: {openrouter.model_name}")

# Тестовый запрос
if os.getenv('OPENROUTER_API_KEY'):
    response = openrouter.query_llm(
        question="Объясни за 50 слов: что такое машинное обучение?",
        temperature=0.7,
        max_tokens=150
    )
    
    print(f"\nОтвет:\n{response}")
else:
    print("\n⚠ Установите OPENROUTER_API_KEY для реального теста")

INFO:faiss.loader:Loading faiss with AVX2 support.
INFO:faiss.loader:Successfully loaded faiss with AVX2 support.


ValueError: OpenRouter API ключ не указан. Установите переменную окружения OPENROUTER_API_KEY или передайте api_key в конструктор.

## 2. VLLM API

Для работы с локально развернутыми моделями.

In [ ]:
from grapharchitect.tools.ApiTools.VLLMTool.VLLMApi import VLLMApi

# Создание клиента
vllm_host = os.getenv("VLLM_HOST", "http://localhost:8000")

vllm_client = VLLMApi(
    vllm_host=vllm_host,
    model_name="Qwen/Qwen2.5-7B-Instruct",
    prompt="You are a helpful assistant."
)

print("VLLM клиент создан")
print(f"Host: {vllm_host}")
print(f"Модель: {vllm_client.model_name}")

# Для теста нужен запущенный VLLM сервер
print("\n⚠ Для теста запустите VLLM:")
print("  vllm serve Qwen/Qwen2.5-7B-Instruct --port 8000")

# Раскомментируйте для реального вызова:
# response = vllm_client.query_llm("Привет, как дела?")
# print(f"Ответ: {response}")

## 3. Infinity Embedder

Для получения качественных семантических эмбеддингов.

In [ ]:
from grapharchitect.tools.ApiTools.InfinityTool.Embedder import InfinityEmbedder

# Создание клиента
infinity_url = os.getenv("INFINITY_BASE_URL", "http://localhost:7997")

infinity = InfinityEmbedder(base_url=infinity_url)

print("Infinity клиент создан")
print(f"URL: {infinity_url}")

# Для теста нужен запущенный Infinity сервер
print("\n⚠ Для теста запустите Infinity:")
print("  docker run -d -p 7997:7997 michaelf34/infinity:latest --model-name BAAI/bge-m3")

# Раскомментируйте для реального вызова:
# result = infinity.get_embedding("Пример текста для векторизации")
# if "embedding" in result:
#     print(f"Эмбеддинг получен, размерность: {len(result['embedding'])}")
# else:
#     print(f"Ошибка: {result}")

## 4. Интеграция с GraphArchitect

Использование API инструментов в GraphArchitect workflow.

In [ ]:
# Создание инструмента на основе OpenRouter
from grapharchitect.entities.base_tool import BaseTool
from grapharchitect.entities.connectors.connector import Connector

class OpenRouterTool(BaseTool):
    """Инструмент на основе OpenRouter."""
    
    def __init__(self, openrouter_client):
        super().__init__()
        self.metadata.tool_name = "OpenRouter GPT-3.5"
        self.metadata.reputation = 0.90
        self._client = openrouter_client
        
        self.input = Connector("text", "question")
        self.output = Connector("text", "answer")
    
    def execute(self, input_data):
        if not os.getenv('OPENROUTER_API_KEY'):
            return "[Заглушка] Ответ без API ключа"
        
        response = self._client.query_llm(
            question=str(input_data),
            temperature=0.7,
            max_tokens=500
        )
        return response

# Создание инструмента
if os.getenv('OPENROUTER_API_KEY'):
    tool = OpenRouterTool(openrouter)
    
    # Тест выполнения
    result = tool.execute("Что такое NLI?")
    print(f"Результат инструмента:\n{result}")
else:
    print("Для теста установите OPENROUTER_API_KEY")

## Заключение

Все API инструменты протестированы и готовы к использованию в GraphArchitect.

**Следующие шаги**:
1. Установите необходимые API ключи
2. Запустите серверы (VLLM, Infinity)
3. Используйте инструменты в workflow

**См. также**: `02_test_local_tools.ipynb` для тестирования локальных инструментов